# This file contains code on multiple regressions 


In [1]:
import pandas as pd

url = "https://gksmyth.github.io/ozdasl/oz/ais.txt"
df = pd.read_csv(url, sep=r"\s+")
df.head()

,Sex,Sport,RCC,WCC,Hc,Hg,Ferr,BMI,SSF,%Bfat,LBM,Ht,Wt
0,female,BBall,3.96,7.5,37.5,12.3,60,20.56,109.1,19.75,63.32,195.9,78.9
1,female,BBall,4.41,8.3,38.2,12.7,68,20.67,102.8,21.30,58.55,189.7,74.4
2,female,BBall,4.14,5.0,36.4,11.6,21,21.86,104.6,19.88,55.36,177.8,69.1
3,female,BBall,4.11,5.3,37.3,12.6,69,21.88,126.4,23.66,57.18,185.0,74.9
4,female,BBall,4.45,6.8,41.5,14.0,29,18.96,80.3,17.64,53.20,184.6,64.6


In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Use every available variable except the response; statsmodels encodes categories.
response = "LBM"
predictors = [column for column in df.columns if column != response]
model_data = df[[response, *predictors]].dropna().copy()
formula = f'{response} ~ ' + " + ".join(
    f"C({column})" if model_data[column].dtype == "object" else column
    for column in predictors
)
model = smf.ols(formula, data=model_data).fit()

print(f"n = {len(model_data)}, predictors = {len(predictors)}")
print(model.summary())

# Assumption and collinearity diagnostics.
residuals = model.resid
shapiro_stat, shapiro_p = stats.shapiro(residuals)
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(residuals, model.model.exog)

numeric_predictors = model_data.select_dtypes(include="number").drop(columns=response)
vif = pd.DataFrame({
    "variable": numeric_predictors.columns,
    "VIF": [variance_inflation_factor(sm.add_constant(numeric_predictors).values, i + 1)
            for i in range(numeric_predictors.shape[1])]
}).sort_values("VIF", ascending=False)

diagnostics = pd.Series({
    "Shapiro-Wilk p-value (normal residuals)": shapiro_p,
    "Breusch-Pagan p-value (constant variance)": bp_f_p,
    "Durbin-Watson (independent residuals)": durbin_watson(residuals),
    "Condition number": model.condition_number,
})
print("\nDiagnostics:")
print(diagnostics)
print("\nNumeric predictor VIF:")
print(vif)

# A held-out check of out-of-sample predictive performance.
train, test = train_test_split(model_data, test_size=0.2, random_state=42)
train_model = smf.ols(formula, data=train).fit()
test_predictions = train_model.predict(test)
print("\nHeld-out test metrics:")
print(pd.Series({
    "R2": r2_score(test[response], test_predictions),
    "RMSE": np.sqrt(mean_squared_error(test[response], test_predictions)),
    "MAE": mean_absolute_error(test[response], test_predictions),
}))

print("\nInterpretation guide: p < 0.05 for Shapiro/Breusch-Pagan indicates a possible assumption violation; VIF > 5 suggests notable multicollinearity.")

AssertionError: 